## **The purpose of this notebook is to compare the similarity of LEiDA states obtained from the Schaefer-100 atlas against the Schaefer-200 and Schaefer-400 results.**

In [ ]:
# Import libraries for data handling and serialization
import pickle
import pandas as pd
import numpy as np


In [ ]:
# Import surface mapping and spatial null/stat comparison tools
from enigmatoolbox.utils.parcellation import parcel_to_surface
from neuromaps import nulls, stats


In [ ]:
# Load K-means cluster centroids from a pickled LEiDA model (K=5)
def read_centroids(result_path):
    model_path = f"{result_path}/clustering/models/model_k_5.pkl"
    with open(model_path, 'rb') as f:
        kmeans_model = pickle.load(f)
    centroids = kmeans_model.cluster_centers_
    return centroids


In [ ]:
# Map centroid values (Limbic + Default ROIs only) to fsa5 surface for visualization
def centroids_to_surf(values, schaefer):
    # Load atlas and create a full ROI × value DataFrame (NaN for non-selected ROIs)
    atlas = pd.read_csv(f"/data/dy/atlas/upgrade/Schaefer{schaefer}x7_MNI.csv")
    df = pd.DataFrame(
        index = atlas['Name'].tolist(), 
        columns = ['values'],
        dtype = float
    )
    
    # Keep only Limbic and Default network ROIs
    rois = atlas[atlas["ICN"] == 'Limbic']["Name"].tolist() + atlas[atlas["ICN"] == 'Default']["Name"].tolist()
    
    df.loc[rois,'values'] = values
    df = df.values.flatten()
    # Parcel values to fsa5 surface
    surf = parcel_to_surface(df,f"schaefer_{schaefer}_fsa5")

    return surf


In [ ]:
# Load centroids from Schaefer 100, 200, and 400 parcellations and map each to surface
centroids_1 = read_centroids(
    result_path = "/home/duyu/2026/tTIS_MDD/LEiDA/LIM_DMN_schaefer100/LEiDA_results"
)
surf1 = np.array([centroids_to_surf(values, 100) for values in centroids_1])

centroids_2 = read_centroids(
    result_path = "/home/duyu/2026/tTIS_MDD/LEiDA/LIM_DMN_schaefer200/LEiDA_results"
)
surf2 = np.array([centroids_to_surf(values, 200) for values in centroids_2])

centroids_4 = read_centroids(
    result_path = "/home/duyu/2026/tTIS_MDD/LEiDA/LIM_DMN_schaefer400/LEiDA_results"
)
surf4 = np.array([centroids_to_surf(values, 400) for values in centroids_4])


In [ ]:
# Compare state centroids across parcellation resolutions using spin-permutation tests
result_2 = pd.DataFrame()
result_4 = pd.DataFrame()

for i1,s1 in enumerate(surf1):
    i1 = f"State{i1+1}"
    # Generate null distribution via Alexander-Bloch spin test (1000 permutations)
    null = nulls.alexander_bloch(s1, atlas='fsaverage', density='10k',n_perm=1000)
    
    # Correlate schaefer-100 state with each schaefer-200 state
    for i2,s2 in enumerate(surf2):
        i2 = f"State{i2+1}"
        r, p = stats.compare_images(s1, s2, nulls=null)
        result_2.loc[f"{i1}_{i2}", ['state1','state2','r','p']] = i1, i2, r, p
        
    # Correlate schaefer-100 state with each schaefer-400 state
    for i4,s4 in enumerate(surf4):
        i4 = f"State{i4+1}"
        r, p = stats.compare_images(s1, s4, nulls=null)
        result_4.loc[f"{i1}_{i4}", ['state1','state4','r','p']] = i1, i4, r, p

# Save results to CSV
result_2.to_csv(f"Corr-100-and_200.csv")
result_4.to_csv(f"Corr-100-and_400.csv")
